# Lab 12: Topic Clustering of News Articles

**Task:** group news articles by topic, without telling the algorithm what the topics are.

**Dataset:** AG News (Kaggle: https://www.kaggle.com/datasets/amananandrai/ag-news-classification-dataset), included as `data/ag_news_classification_dataset.zip` as downloaded: 120,000 training and 7,600 test news articles (title and short description) in four classes: World, Sports, Business, Sci/Tech. We use the class only at the end, to check what the clusters found.

**What we will do:**
1. Turn the text into TF-IDF vectors and discuss cosine similarity.
2. Cluster with K-means ($k = 4$, the known number of topics, then other $k$), and with hierarchical clustering on a small subset (dendrogram).
3. Validate with the silhouette score and by reading the top terms of each cluster.
4. For each cluster: its top 15 keywords and 5 representative headlines.

In [ ]:
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 150)
pd.set_option("display.max_colwidth", 120)
z = zipfile.ZipFile("data/ag_news_classification_dataset.zip")
print(z.namelist())
news = pd.read_csv(z.open("train.csv"))
print(news.shape)
news.head()

The classes are coded 1–4: 1 = World, 2 = Sports, 3 = Business, 4 = Sci/Tech (from the dataset description). We take a random sample of 20,000 articles to keep the lab fast; title and description together form the document.

The descriptions contain some leftovers: HTML entities (`&lt;b&gt;`, `#36;` for `$`, and `quot` left over from `&quot;`), backslashes where line breaks were, and source tags such as "(Reuters)" and "AFP -". Source names appear in every topic, so they would only add noise to the clusters; we remove the most common ones.

In [ ]:
topic_names = {1: "World", 2: "Sports", 3: "Business", 4: "Sci/Tech"}
docs = news.sample(20_000, random_state=0).reset_index(drop=True)
docs["topic"] = docs["Class Index"].map(topic_names)
docs["text"] = (docs["Title"] + ". " + docs["Description"])
docs["clean"] = (docs["text"].str.lower()
                 .str.replace(r"&lt;.*?&gt;|&\w+;|#\d+;", " ", regex=True)
                 .str.replace(r"\b(quot|lt|gt|amp|reuters|afp|ap|cnn|ziff davis|space\.com|newsfactor|techweb)\b", " ", regex=True)
                 .str.replace(r"[^a-z\s]", " ", regex=True))
print(docs["topic"].value_counts().to_dict())
print(docs.loc[0, "text"][:200]); print(docs.loc[0, "clean"][:200])

## 1. TF-IDF vectors and cosine similarity

Each article becomes a TF-IDF vector: one entry per vocabulary word, the word's frequency in the article times its inverse document frequency (rare words weigh more, words in every article weigh almost nothing), scaled to length 1. We remove English stopwords (for *topics*, unlike sentiment, words like "not" carry little), and ignore words in fewer than 5 articles or in more than half of them.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words="english", min_df=5, max_df=0.5, sublinear_tf=True)
X = vectorizer.fit_transform(docs["clean"])
terms = vectorizer.get_feature_names_out()
print(X.shape, f"| average non-zero words per article: {X.nnz / X.shape[0]:.0f}")

**Cosine similarity** measures the angle between two document vectors: $\cos(\mathbf{a}, \mathbf{b}) = \dfrac{\mathbf{a}^T\mathbf{b}}{\|\mathbf{a}\|\,\|\mathbf{b}\|}$. For TF-IDF vectors it is between 0 (no words in common) and 1 (the same words in the same proportions). It ignores length: a long article and a short one on the same subject can be very similar. Since our vectors already have length 1, cosine similarity is just the dot product, and the squared Euclidean distance is $\|\mathbf{a} - \mathbf{b}\|^2 = 2 - 2\cos(\mathbf{a}, \mathbf{b})$: Euclidean K-means on normalised TF-IDF vectors groups documents by cosine similarity.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

query = 3
sims = cosine_similarity(X[query], X).ravel()
print("article:", docs.loc[query, "text"][:120])
for i in sims.argsort()[::-1][1:5]:
    print(f"  cos {sims[i]:.2f} | {docs.loc[i, 'topic']:8s} | {docs.loc[i, 'text'][:100]}")

## 2. K-means

### k = 4

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

km4 = KMeans(n_clusters=4, n_init=10, random_state=0).fit(X)
docs["cluster"] = km4.labels_
print("cluster sizes:", np.bincount(km4.labels_))

def top_terms(model, n=15):
    order = model.cluster_centers_.argsort()[:, ::-1]
    return {c: [terms[i] for i in order[c, :n]] for c in range(model.n_clusters)}

for c, words in top_terms(km4).items():
    print(f"cluster {c}: {', '.join(words)}")

The top terms of each centroid (the words with the largest average TF-IDF weight in the cluster) let us name the clusters. Do they match the four real topics? (The clustering never saw them.)

In [ ]:
from sklearn.metrics import adjusted_rand_score
print(pd.crosstab(docs["cluster"], docs["topic"]))
print("adjusted Rand index vs the true topics:", round(adjusted_rand_score(docs["topic"], docs["cluster"]), 3))

Not great. Two clusters are clear (oil and markets; Iraq and world politics), one mixes business and technology companies, and one huge cluster holds all the sports *plus* thousands of articles of every other kind. The **adjusted Rand index** (ARI) measures agreement with the true topics: 1 is perfect, 0 is no better than chance; 0.26 is weak.

Why? Each article has only about 18 non-zero words out of 10,000. Two articles about the same topic often share *no* words at all ("striker" and "goalkeeper" never meet), so in this space most articles are almost equally far from everything, and the centroids are pulled around by a few frequent words.

### A fix: reduce the dimensions first (LSA)
**Truncated SVD** is PCA for sparse matrices (it works on the TF-IDF matrix directly, without centring). Applied to text it is known as **latent semantic analysis (LSA)**: each of the new dimensions is a weighted combination of words that tend to appear together, a kind of "theme". Two sports articles with no words in common still load on the same sports-like dimensions. We keep 50 dimensions and re-normalise each document vector to length 1, so that K-means still compares documents by cosine.

In [ ]:
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer

svd = TruncatedSVD(n_components=50, random_state=0).fit(X)
print("variance explained by 50 dimensions:", round(svd.explained_variance_ratio_.sum(), 3))
L = Normalizer().fit_transform(svd.transform(X))
kml = KMeans(n_clusters=4, n_init=10, random_state=0).fit(L)
docs["cluster"] = kml.labels_
print(pd.crosstab(docs["cluster"], docs["topic"]))
print("adjusted Rand index:", round(adjusted_rand_score(docs["topic"], docs["cluster"]), 3))

Much better: balanced clusters that correspond to the topics (ARI 0.55, twice as high), even though 50 dimensions keep only a small part of the variance of the 10,000-word space: they keep the part that matters for topics. Business and Sci/Tech still overlap (technology companies' earnings belong to both).

To read the clusters we need words again. For each cluster we average the original TF-IDF vectors of its articles, which gives a centroid in word space:

In [ ]:
def word_centroids(labels):
    return np.vstack([np.asarray(X[labels == c].mean(axis=0)).ravel() for c in range(labels.max() + 1)])

def top_terms_from(labels, n=15):
    cents = word_centroids(labels)
    return {c: [terms[i] for i in cents[c].argsort()[::-1][:n]] for c in range(len(cents))}

for c, words in top_terms_from(kml.labels_).items():
    print(f"cluster {c}: {', '.join(words)}")

### Exploring k
The silhouette score (with the cosine distance) on the LSA vectors, and the agreement with the true topics, for several $k$:

In [ ]:
rows = []
for k in [2, 3, 4, 5, 6, 8, 10, 15]:
    km = KMeans(n_clusters=k, n_init=5, random_state=0).fit(L)
    rows.append({"k": k, "SSE": km.inertia_,
                 "silhouette (cosine)": silhouette_score(L, km.labels_, metric="cosine", sample_size=4000, random_state=0),
                 "ARI vs topics": adjusted_rand_score(docs["topic"], km.labels_)})
explore = pd.DataFrame(rows)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(explore["k"], explore["SSE"], "o-"); axes[0].set_title("SSE"); axes[0].set_xlabel("k")
axes[1].plot(explore["k"], explore["silhouette (cosine)"], "o-"); axes[1].set_title("silhouette (cosine)"); axes[1].set_xlabel("k")
plt.show()
explore.round(4)

Silhouette scores on text are low (documents form a continuum, not tight separate groups), and neither curve points clearly at one $k$. With more clusters, K-means splits the broad topics into sub-topics, which can be more useful than the four official classes:

In [ ]:
km8 = KMeans(n_clusters=8, n_init=10, random_state=0).fit(L)
for c, words in top_terms_from(km8.labels_, 10).items():
    print(f"cluster {c} ({np.sum(km8.labels_ == c)} articles): {', '.join(words)}")

## 3. Hierarchical clustering on a small subset

Hierarchical clustering needs all pairwise distances, which is impractical for 20,000 documents, so we use 60 articles (15 per topic) and draw the dendrogram, on the 50-dimensional LSA vectors, with cosine distance and average linkage.

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

small = docs.groupby("topic", group_keys=False).sample(15, random_state=1)
Z = linkage(L[small.index], method="average", metric="cosine")
plt.figure(figsize=(14, 5))
dendrogram(Z, labels=[f"{t[:5]}: {s[:25]}" for t, s in zip(small["topic"], small["Title"])], leaf_font_size=7, color_threshold=0.9)
plt.title("hierarchical clustering of 60 articles (average linkage, cosine distance)"); plt.ylabel("cosine distance"); plt.tight_layout(); plt.show()
cut = fcluster(Z, t=4, criterion="maxclust")
print(pd.crosstab(cut, small["topic"].values, rownames=["cluster"], colnames=["topic"]))

**How the dendrogram works:** each article starts as its own cluster; at each step the two closest clusters merge, with "closest" meaning the smallest *average* cosine distance between their articles (average linkage). The height of a merge is that distance. Articles on the same story merge low down (very similar wording), and you can see small branches of same-topic articles. Cutting the whole tree into 4 clusters, though, gives a poor grouping here (see the table): with average linkage on only 60 short articles, one large cluster absorbs most of them while a few small clusters hold tight groups or single articles. Hierarchical clustering is most useful for *exploring* a small collection (which articles are near-duplicates, which form a story), less for a balanced partition; for that, K-means on the LSA vectors did much better.

## 4. Deliverables: keywords and representative headlines

For each cluster of the LSA K-means model with $k = 4$: the 15 words with the largest average TF-IDF weight, and the 5 headlines closest to the cluster centre (the most typical articles).

In [ ]:
centre_sim = L @ kml.cluster_centers_.T
for c, words in top_terms_from(kml.labels_, 15).items():
    members = np.where(kml.labels_ == c)[0]
    closest = members[np.argsort(centre_sim[members, c])[::-1][:5]]
    print(f"=== Cluster {c}: {len(members)} articles, mostly {docs.loc[members, 'topic'].value_counts().idxmax()}")
    print("keywords:", ", ".join(words))
    for i in closest:
        print("   -", docs.loc[i, "Title"])
    print()

**Summary.** K-means directly on sparse TF-IDF vectors produced one huge catch-all cluster; reducing the vectors to 50 LSA dimensions first gave four balanced clusters that match the real topics far better (ARI 0.55 against 0.26), without the algorithm ever seeing the topics. Sports separates best; Business and Sci/Tech overlap, since technology companies' results belong to both. The silhouette score is low for text and doesn't identify a "right" $k$; reading the top terms and typical headlines is how to judge text clusters, and larger $k$ reveals sub-topics (oil prices, Iraq, elections, software, earnings).